# Generative Adversarial Networks

**AI2 · Computer Vision** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

In the previous section, we learned about **generative models**: models that can generate new images similar to the ones in the training dataset. VAE was a good example of a generative model.


However, if we try to generate something really meaningful, like a painting at reasonable resolution, with VAE, we will see that training does not converge well. For this use case, we should learn about another architecture specifically targeted at generative models - **Generative Adversarial Networks**, or GANs.

The main idea of a GAN is to have two neural networks that will be trained against each other:

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/10-GANs/images/gan_architecture.png)

> Image by [Dmitry Soshnikov](http://soshnikov.com)

> ✅ A little vocabulary:
> * **Generator** is a network that takes some random vector, and produces the image as a result
> * **Discriminator** is a network that takes an image, and it should tell whether it is a real image (from training dataset), or it was generated by a generator. It is essentially an image classifier.



## Discriminator

The architecture of discriminator does not differ from an ordinary image classification network. In the simplest case it can be fully-connected classifier, but most probably it will be a [convolutional network](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/07-ConvNets/README.md).

> ✅ A GAN based on convolutional networks is called a [DCGAN](https://arxiv.org/pdf/1511.06434.pdf)

A CNN discriminator consists of the following layers: several convolutions+poolings (with decreasing spatial size) and, one-or-more fully-connected layers to get "feature vector", final binary classifier.

> ✅ A 'pooling' in this context is a technique that reduces the size of the image. "Pooling layers reduce the dimensions of data by combining the outputs of neuron clusters at one layer into a single neuron in the next layer." - [source](https://wikipedia.org/wiki/Convolutional_neural_network#Pooling_layers)



## Generator

A Generator is slightly more tricky. You can consider it to be a reversed discriminator. Starting from a latent vector (in place of a feature vector), it has a fully-connected layer to convert it into the required size/shape, followed by deconvolutions+upscaling. This is similar to *decoder* part of [autoencoder](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/09-Autoencoders/README.md).

> ✅ Because the convolution layer is implemented as a linear filter traversing the image, deconvolution is essentially similar to convolution, and can be implemented using the same layer logic.

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/4-ComputerVision/10-GANs/images/gan_arch_detail.png)

> Image by [Dmitry Soshnikov](http://soshnikov.com)



## Training the GAN

GANs are called **adversarial** because there is a constant competition between the generator and the discriminator. During this competition, both generator and discriminator improve, thus the network learns to produce better and better pictures.

The training happens in two stages:

* **Training the discriminator**. This task is pretty straightforward: we generate a batch of images by the generator, labeling them 0, which stands for fake image, and taking a batch of images from the input dataset (with label 1, real image). We obtain some *discriminator loss*, and perform backprop.
* **Training the generator**. This is slightly more tricky, because we do not know the expected output for the generator directly. We take the whole GAN network consisting of a generator followed by discriminator, feed it with some random vectors, and expect the result to be 1 (corresponding to real images). We then freeze the parameters of the discriminator (we do not want it to be trained at this step), and perform the backprop.

During this process, both the generator and the discriminator losses are not going down significantly. In the ideal situation, they should oscillate, corresponding to both networks improving their performance.



## Exercises: GANs

* [GAN Notebook in TensorFlow/Keras](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/GANTF.ipynb)
* [GAN Notebook in PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/GANPyTorch.ipynb)



## Problems with GAN training

GANs are known to be especially difficult to train. Here are a few problems:

* **Mode Collapse**. By this term we mean that the generator learns to produce one successful image that tricks the generator, and not a variety of different images.
* **Sensitivity to hyperparameters**. Often you can see that a GAN does not converge at all, and then suddenly decreases in the learning rate leading to convergence.
* Keeping a **balance** between the generator and the discriminator. In many cases discriminator loss can drop to zero relatively quickly, which results in the generator being unable to train further. To overcome this, we can try setting different learning rates for the generator and discriminator, or skip discriminator training if the loss is already too low.
* Training for **high resolution**. Reflecting the same problem as with autoencoders, this problem is triggered because reconstructing too many layers of convolutional network leads to artifacts. This problem is typically solved with so-called **progressive growing**, when first a few layers are trained on low-res images, and then layers are "unblocked" or added. Another solution would be adding extra connections between layers and training several resolutions at once - see this [Multi-Scale Gradient GANs paper](https://arxiv.org/abs/1903.06048) for details.



## Style Transfer

GANs is a great way to generate artistic images. Another interesting technique is so-called **style transfer**, which takes one **content image**, and re-draws it in a different style, applying filters from **style image**. 

The way it works is the following:
* We start with a random noise image (or with a content image, but for the sake of understanding it is easier to start from random noise)
* Our goal would be to create such an image, that would be close to both content image and style image. This would be determined by two loss functions:
   - **Content loss** is computed based on the features extracted by the CNN at some layers from current image and content image
   - **Style loss** is computed between current image and style image in a clever way using Gram matrices (more details in the [example notebook](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/StyleTransfer.ipynb))
* To make the image smoother and remove noise, we also introduce **Variation loss**, which computes average distance between neighboring pixels
* The main optimization loop adjusts current image using gradient descent (or some other optimization algorithm) to minimize the total loss, which is a weighted sum of all three losses.



## Conclusion

In this lesson, you learned about GANS and how to train them. You also learned about the special challenges that this type of Neural Network can face, and some strategies on how to move past them.



## Challenge

Run through the [Style Transfer notebook](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/StyleTransfer.ipynb) using your own images.



## Review & Self Study

For reference, read more about GANs in these resources:

* Marco Pasini, [10 Lessons I Learned Training GANs for one Year](https://towardsdatascience.com/10-lessons-i-learned-training-generative-adversarial-networks-gans-for-a-year-c9071159628)
* [StyleGAN](https://en.wikipedia.org/wiki/StyleGAN), a *de facto* GAN architecture to consider
* [Creating Generative Art using GANs on Azure ML](https://soshnikov.com/scienceart/creating-generative-art-using-gan-on-azureml/)



## Assignment

Revisit one of the two notebooks associated to this lesson and retrain the GAN on your own images. What can you create?



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

1. [Spherical latent interpolation](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-slerp)
2. [One forward-diffusion step](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-q-sample)
3. [Linear beta schedule](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-beta-schedule)
4. [One deterministic DDIM step](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-ddim-step)

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Spherical latent interpolation

●●● · ~30 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-slerp)

> Linear interpolation between latent vectors cuts through empty regions of latent space; spherical interpolation (slerp) follows the great-circle arc. For Ω = angle(u, v): slerp(u,v,t) = [sin((1−t)Ω)·u + sin(tΩ)·v] / sinΩ. At t=0 and t=1 it reproduces u and v exactly; when sinΩ ≈ 0 (parallel vectors) fall back to plain lerp.

Implement `slerp(u, v, t)` → interpolated vector. Let `cos = (u·v)/(|u|·|v|)` clipped to [-1, 1], `omega = arccos(cos)`. If `sin(omega) < 1e-8` return the linear blend `(1-t)*u + t*v`; otherwise return `(sin((1-t)*omega)*u + sin(t*omega)*v) / sin(omega)`.

<details><summary>Hint</summary>

Compute omega via math.acos on the clipped cosine; the two sin() coefficients weight u and v, divided by sin(omega).

</details>


In [ ]:
import numpy as np

def slerp(u, v, t):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    # your code
    return u

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(3, 8))\n    u = rng.normal(0, 1, d)\n    if np.linalg.norm(u) < 1e-6:\n        u = np.ones(d)\n    uh = u / np.linalg.norm(u)\n    w = None\n    for cand in (rng.normal(0, 1, d), np.ones(d), np.arange(1.0, d + 1)):\n        p = cand - np.dot(cand, uh) * uh\n        if np.linalg.norm(p) > 1e-6:\n            w = p / np.linalg.norm(p)\n            break\n    r_case = rng.random()\n    if r_case < 0.15:\n        v = u * float(rng.uniform(0.5, 2.0))    # parallel — sin(Omega) = 0, must hit the lerp fallback\n    elif r_case < 0.25:\n        v = -u * float(rng.uniform(0.5, 2.0))   # anti-parallel — same degenerate branch\n    else:\n        theta = float(rng.uniform(0.3, 2.5))\n        nv = float(rng.uniform(0.5, 2.0))\n        v = nv * (np.cos(theta) * uh + np.sin(theta) * w)\n    return {\"u\": u, \"v\": v, \"t\": float(rng.uniform(0.0, 1.0))}\n\ndef slerp(u, v, t):\n    u = np.asarray(u, dtype=float)\n    v = np.asarray(v, dtype=float)\n    cos = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))\n    omega = math.acos(max(-1.0, min(1.0, float(cos))))\n    so = math.sin(omega)\n    if so < 1e-8:\n        return (1.0 - t) * u + t * v\n    return (math.sin((1.0 - t) * omega) * u + math.sin(t * omega) * v) / so", "slerp", 20)

### Task 2 — One forward-diffusion step

●○○ · ~8 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-q-sample)

> The forward diffusion process adds Gaussian noise step by step, but the closed form jumps straight to step t: x_t = √ᾱ_t · x_0 + √(1−ᾱ_t)·ε with ε ~ N(0, I) and ᾱ_t = Π_{s≤t}(1−β_s). No loop needed — this is how training samples noisy x_t directly from clean x_0.

Implement `q_sample(x0, abar, eps)` → `sqrt(abar) * x0 + sqrt(1 - abar) * eps`, where `abar` is the scalar cumulative product ᾱ_t and `eps` is the given standard normal noise of the same shape as `x0`.

<details><summary>Hint</summary>

math.sqrt(abar) * x0 + math.sqrt(1.0 - abar) * eps — one line.

</details>


In [ ]:
import numpy as np

def q_sample(x0, abar, eps):
    x0 = np.asarray(x0, dtype=float)
    # your code
    return x0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    m = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))\n    return {\"x0\": rng.normal(0, 1, (m, d)),\n            \"abar\": float(rng.uniform(0.02, 0.95)),\n            \"eps\": rng.normal(0, 1, (m, d))}\n\ndef q_sample(x0, abar, eps):\n    x0 = np.asarray(x0, dtype=float)\n    eps = np.asarray(eps, dtype=float)\n    return math.sqrt(abar) * x0 + math.sqrt(1.0 - abar) * eps", "q_sample", 20)

### Task 3 — Linear beta schedule

●●○ · ~10 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-beta-schedule)

> A diffusion noise schedule picks variances β_1..β_T — commonly linearly spaced from ~1e-4 to ~0.02 — and precomputes α_t = 1−β_t and ᾱ_t = Π_{s≤t} α_s. Every closed-form sampling formula uses ᾱ, so computing the cumulative product once is the standard first step.

Implement `beta_schedule(T, b0, b1)` → dict with keys `"betas"` (T values linearly spaced from `b0` to `b1`, inclusive endpoints), `"alphas"` (1 − betas) and `"alphas_cumprod"` (cumulative product of alphas).

<details><summary>Hint</summary>

betas = np.linspace(b0, b1, T); alphas = 1 - betas; alphas_cumprod = np.cumprod(alphas).

</details>


In [ ]:
import numpy as np

def beta_schedule(T, b0, b1):
    betas = np.linspace(b0, b1, T)
    # your code
    return {"betas": betas}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    return {\"T\": int(rng.integers(5, 30)),\n            \"b0\": float(rng.uniform(0.0001, 0.005)),\n            \"b1\": float(rng.uniform(0.01, 0.05))}\n\ndef beta_schedule(T, b0, b1):\n    betas = np.linspace(b0, b1, T)\n    alphas = 1.0 - betas\n    return {\"betas\": betas, \"alphas\": alphas,\n            \"alphas_cumprod\": np.cumprod(alphas)}", "beta_schedule", 20)

### Task 4 — One deterministic DDIM step

●●● · ~30 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m10-ddim-step)

> DDPM adds fresh Gaussian noise at every step — slow and stochastic. DDIM removes noise **deterministically** (η = 0). The network sees the noisy image x_t and returns a noise estimate ε̂; the coefficient ᾱ_t is how much clean signal remains at step t, and ᾱ_t < ᾱ_{t−1}. Step one: predict the clean sample, x̂0 = (x_t − √(1−ᾱ_t)·ε̂)/√ᾱ_t. Step two: re-noise to the previous level, x_{t−1} = √ᾱ_{t−1}·x̂0 + √(1−ᾱ_{t−1})·ε̂. With η = 0 the same ε̂ is reused both times, so the whole trajectory is a deterministic function of x_T.

Implement `ddim_step(x_t, eps, abar_t, abar_prev)` → the deterministic (η = 0) DDIM update: `x0 = (x_t − sqrt(1 − abar_t)·eps)/sqrt(abar_t)`, then return `sqrt(abar_prev)·x0 + sqrt(1 − abar_prev)·eps`. The scalars satisfy 0 < abar_t < abar_prev < 1.

<details><summary>Hint</summary>

Use math.sqrt for the scalar coefficients — build x0 first, then the convex-looking recombination with eps.

</details>


In [ ]:
import numpy as np

def ddim_step(x_t, eps, abar_t, abar_prev):
    x_t = np.asarray(x_t, dtype=float)
    # your code
    return x_t

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    m = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))\n    abar_prev = float(rng.uniform(0.3, 0.98))\n    abar_t = float(rng.uniform(0.02, abar_prev - 0.01))\n    return {\"x_t\": rng.normal(0, 1, (m, d)), \"eps\": rng.normal(0, 1, (m, d)),\n            \"abar_t\": abar_t, \"abar_prev\": abar_prev}\n\ndef ddim_step(x_t, eps, abar_t, abar_prev):\n    x_t = np.asarray(x_t, dtype=float)\n    eps = np.asarray(eps, dtype=float)\n    x0 = (x_t - math.sqrt(1.0 - abar_t) * eps) / math.sqrt(abar_t)\n    return math.sqrt(abar_prev) * x0 + math.sqrt(1.0 - abar_prev) * eps", "ddim_step", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs)
- [Yandex ML handbook (RU): convolutional networks](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/convolutional-neural-networks)
- [Yandex ML handbook (RU): generative models](https://contest.yandex.ru/tracks/ml/generative-models/introduction-to-generative-modeling)